# Single step forecasting with machine learning

[Forecasting with Machine Learning - Course](https://www.trainindata.com/p/forecasting-with-machine-learning)

In this notebook, we will pick up the table of predictive features and a target from the first notebook and train a machine learning model to do forecasting.

We will do **single step** ahead forecasting.

In [22]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso
from sklearn.metrics import root_mean_squared_error

# Load data

We will use the electricity demand dataset found [here](https://github.com/tidyverts/tsibbledata/tree/master/data-raw/vic_elec/VIC2015).

**Citation:**

Godahewa, Rakshitha, Bergmeir, Christoph, Webb, Geoff, Hyndman, Rob, & Montero-Manso, Pablo. (2021). Australian Electricity Demand Dataset (Version 1) [Data set]. Zenodo. https://doi.org/10.5281/zenodo.4659727

**Description of data:**

A description of the data can be found [here](https://rdrr.io/cran/tsibbledata/man/vic_elec.html). The data contains electricity demand in Victoria, Australia, at 30 minute intervals over a period of 12 years, from 2002 to early 2015. There is also the temperature in Melbourne at 30 minute intervals and public holiday dates.

In [ ]:
# Electricity demand.
url = "https://raw.githubusercontent.com/tidyverts/tsibbledata/master/data-raw/vic_elec/VIC2015/demand.csv"
df = pd.read_csv(url)

df.drop(columns=["Industrial"], inplace=True)

# Convert the integer Date to an actual date with datetime type
df["date"] = df["Date"].apply(
    lambda x: pd.Timestamp("1899-12-30") + pd.Timedelta(x, unit="days")
)

# Create a timestamp from the integer Period representing 30 minute intervals
df["date_time"] = df["date"] + \
    pd.to_timedelta((df["Period"] - 1) * 30, unit="m")

df.dropna(inplace=True)

# Rename columns
df = df[["date_time", "OperationalLessIndustrial"]]

df.columns = ["date_time", "demand"]

# Resample to hourly
df = (
    df.set_index("date_time")
    .resample("h")
    .agg({"demand": "sum"})
)

df.head()

## Lag features

We shift pass values of the time series forward. We discussed these 3 features in a previous notebook / video.

In [ ]:
df[f"y_lag_1"] = df["demand"].shift(periods=1)

freq = "24h"
df[f"y_lag_{freq}"] = df["demand"].shift(freq=freq)

freq = "6d"
df[f"y_lag_{freq}"] = df["demand"].shift(freq=freq)

df.head()

## Window features

We aggregate values within windows in the past.  We discussed these windows in a previous notebook / video.

In [ ]:
# We'll use a 3H window size.

result = (
    df["demand"]
    .rolling(window=3) # Pick window size.
    .agg(["mean", "std"]) # Pick statistics.
    .shift(freq="1h") # Lag by 1 hour to avoid data leakage.
)  

result = result.add_prefix("y_window_3_")

# add features to main dataframe

df = df.merge(result, how="left", left_index=True, right_index=True)

df.head()

In [ ]:
# We'll use a window size of 24 hours to smooth 
# over daily seasonality.

result = (
    df["demand"]
    .rolling(window=24) # Pick window size.
    .agg(["mean", "std"]) # Pick statistics.
    .shift(freq="1h") # Lag by 1 hour to avoid data leakage.
)  

result = result.add_prefix("y_window_24_")

# add features to main dataframe
df = df.merge(result, how="left", left_index=True, right_index=True)

df.head()

## Datetime features

We'll create date and time related features from the time series. 

In [ ]:
df["month"] = df.index.month
df["day"] = df.index.dayofweek
df["hour"] = df.index.hour

df.head()

## Finalize tabularization

In [ ]:
df.dropna(inplace=True)

y = df["demand"]
X = df.drop("demand", axis=1)

# Predictors
X.head()

In [ ]:
# target

y.head()

## Lasso

We'll split the data into train and test.

We train the model on the train set and evaluate it on the test set.

In [9]:
# Split into train and test

# We leave 2015 in the test set

end_train = '2014-12-31 23:59:59'

X_train = X.loc[:end_train]
X_test  = X.loc[end_train:]

y_train = y.loc[:end_train]
y_test  = y.loc[end_train:]

In [ ]:
lasso = Lasso(random_state=9)

lasso.fit(X_train, y_train)

In [ ]:
preds = lasso.predict(X_test)

rmse = root_mean_squared_error(y_test, preds)

print(f"performance of lasso = {np.round(rmse, 0)}")

These results is better than any of the basic forecasts that we did in the previous notebook!

In [ ]:
# convert to series for plotting

preds = pd.Series(preds, index=X_test.index)

preds

In [ ]:
# plot predictions vs actuals

fig, ax = plt.subplots(figsize=(6, 3))
y_train[-100:].plot(ax=ax, label='train')
y_test[:100].plot(ax=ax, label='test')
preds.iloc[:100].plot(ax=ax, label='predictions')
plt.title("Lasso forecasting")
ax.legend(bbox_to_anchor=(1.3, 1.0))

In [ ]:
# We can also understand what is driving the predictions of this
# lasso regression.

importance = lasso.coef_
features = lasso.feature_names_in_

pd.Series(importance, index=features).sort_values(ascending=False).plot.bar()
plt.title("Feature importance")
plt.ylabel("Coefficient value")

The hour and the day are the most important predictors of energy consumption.

In [ ]:
y.groupby(X["day"]).mean().plot(figsize=(8,4))

plt.title("Daily energy demand")
plt.ylabel("Energy demand")
plt.xlabel("day of the week")
plt.tight_layout()

## Random forest

In [ ]:
rf = RandomForestRegressor(n_estimators=10, random_state=9)

rf.fit(X_train, y_train)

In [ ]:
preds = rf.predict(X_test)

rmse = root_mean_squared_error(y_test, preds)

print(f"performance of random forests = {np.round(rmse, 0)}")

These results are even better than those from Lasso!

In [ ]:
# convert to series for plotting

preds = pd.Series(preds, index=X_test.index)

preds

In [ ]:
# plot predictions vs actuals

fig, ax = plt.subplots(figsize=(6, 3))
y_train[-100:].plot(ax=ax, label='train')
y_test[:100].plot(ax=ax, label='test')
preds.iloc[:100].plot(ax=ax, label='predictions')
plt.title("Random forests forecasting")
ax.legend(bbox_to_anchor=(1.3, 1.0));

In [ ]:
importance = rf.feature_importances_
features = rf.feature_names_in_

pd.Series(importance, index=features).sort_values(ascending=False).plot.bar()
plt.title("Feature importance")
plt.ylabel("Feature importance")

For the random forests, the energy consumption in the previews hour, seems to be the most important feature to predict demand.